# LDA

## 1. Import Libraries

In [14]:
import pandas as pd
import numpy as np
import os
import pickle

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import StackingClassifier, RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.preprocessing import PolynomialFeatures
from sklearn.pipeline import Pipeline
from sklearn.model_selection import cross_val_score, GridSearchCV
from sklearn.metrics import (accuracy_score, precision_score, recall_score, 
                             f1_score, roc_auc_score)

import warnings
warnings.filterwarnings('ignore')

## Load Data

In [15]:
data_dir = '../data/'
target_col = 'class'

train_df = pd.read_csv(os.path.join(data_dir, 'train_data.csv'))
test_df  = pd.read_csv(os.path.join(data_dir, 'test_data.csv'))

X_train = train_df.drop(columns=[target_col])
y_train = train_df[target_col]
X_test  = test_df.drop(columns=[target_col])
y_test  = test_df[target_col]

###  Danh sách lưu kết quả để xuất CSV

In [16]:
results_list = []

def evaluate_model(model, name, X_test, y_test):
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]
    
    res = {
        'Algorithm': name,
        'Accuracy':  round(accuracy_score(y_test, y_pred), 2),
        'Precision': round(precision_score(y_test, y_pred), 2),
        'Recall':    round(recall_score(y_test, y_pred), 2),
        'F1-Score':  round(f1_score(y_test, y_pred), 2),
        'AUC':       round(roc_auc_score(y_test, y_prob), 2)
    }
    
    results_list.append(res)
    print(f"[{name}] Acc: {res['Accuracy']} | Prec: {res['Precision']} | "
          f"Rec: {res['Recall']} | F1: {res['F1-Score']} | AUC: {res['AUC']}")
    return model

### 3. CHẠY BASELINE MODEL (Tham số mặc định)

In [17]:
print("V1: Baseline LDA")
v1_model = LinearDiscriminantAnalysis()
v1_model.fit(X_train, y_train)
evaluate_model(v1_model, "V1: LDA Baseline", X_test, y_test)

V1: Baseline LDA
[V1: LDA Baseline] Acc: 0.99 | Prec: 0.97 | Rec: 1.0 | F1: 0.99 | AUC: 1.0


,solver,'svd'
,shrinkage,None
,priors,None
,n_components,None
,store_covariance,False
,tol,0.0001
,covariance_estimator,None


### 4. CHẠY TUNING MODEL (Dùng GridSearchCV)

In [18]:
print("\n--- V2: GridSearchCV Tuning ---")
# Lưu ý: shrinkage chỉ dùng được với solver='lsqr' hoặc 'eigen'
param_grid = [
    {'solver': ['svd']},  # svd không hỗ trợ shrinkage
    {'solver': ['lsqr', 'eigen'], 
     'shrinkage': [None, 'auto', 0.1, 0.3, 0.5, 0.7, 0.9]}
]

grid_search = GridSearchCV(
    LinearDiscriminantAnalysis(),
    param_grid,
    cv=5,
    scoring='accuracy',
    n_jobs=-1
)

grid_search.fit(X_train, y_train)
v2_model = grid_search.best_estimator_

print(f"Best Params: {grid_search.best_params_}")
evaluate_model(v2_model, "V2: LDA Tuned (GridSearch)", X_test, y_test)


--- V2: GridSearchCV Tuning ---
Best Params: {'solver': 'svd'}
[V2: LDA Tuned (GridSearch)] Acc: 0.99 | Prec: 0.97 | Rec: 1.0 | F1: 0.99 | AUC: 1.0


,solver,'svd'
,shrinkage,None
,priors,None
,n_components,None
,store_covariance,False
,tol,0.0001
,covariance_estimator,None


In [19]:
print("\n--- V3: Ensemble Learning (Stacking) ---")
base_estimators = [
    ('lda', v2_model),  # LDA đã tune làm base
    ('rf',  RandomForestClassifier(n_estimators=50, random_state=42))
]

v3_model = StackingClassifier(
    estimators=base_estimators,
    final_estimator=LogisticRegression(random_state=42),
    cv=5
)

v3_model.fit(X_train, y_train)
evaluate_model(v3_model, "V3: LDA Stacking Ensemble", X_test, y_test)


--- V3: Ensemble Learning (Stacking) ---
[V3: LDA Stacking Ensemble] Acc: 0.99 | Prec: 0.97 | Rec: 1.0 | F1: 0.99 | AUC: 1.0


,estimators,"[('lda', ...), ('rf', ...)]"
,final_estimator,LogisticRegre...ndom_state=42)
,cv,5
,stack_method,'auto'
,n_jobs,None
,passthrough,False
,verbose,0
,solver,'svd'
,shrinkage,None
,priors,None
,n_components,None


### (5) Lưu kết quả

In [20]:
save_path = '../experiment/LDA/'
os.makedirs(save_path, exist_ok=True)

df_results = pd.DataFrame(results_list)
csv_output = os.path.join(save_path, 'lda_evaluation_results.csv')
df_results.to_csv(csv_output, index=False)

print("\n" + "-" * 40)
print(f"Đã lưu CSV tại: {csv_output}")
print("-" * 40)
display(df_results)


----------------------------------------
Đã lưu CSV tại: ../experiment/LDA/lda_evaluation_results.csv
----------------------------------------


,Algorithm,Accuracy,Precision,Recall,F1-Score,AUC
0,V1: LDA Baseline,0.99,0.97,1.0,0.99,1.0
1,V2: LDA Tuned (GridSearch),0.99,0.97,1.0,0.99,1.0
2,V3: LDA Stacking Ensemble,0.99,0.97,1.0,0.99,1.0


In [21]:
!jupyter nbconvert --to html LDA.ipynb

[NbConvertApp] Converting notebook LDA.ipynb to html
[NbConvertApp] Writing 366277 bytes to LDA.html
